In [6]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.processing import WESADDataset
from src.features import process_subject_data
from src.models import train_and_evaluate_loso

In [7]:
# S12 is excluded from WESAD due to data collection issues (documented in the paper)
ALL_SUBJECTS = [
    'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8',
    'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17'
]

loader = WESADDataset('../data/WESAD')

print(f"Subjects to process: {len(ALL_SUBJECTS)}")
print(ALL_SUBJECTS)

Subjects to process: 15
['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']


In [8]:
all_features = []
failed = []

for subject in ALL_SUBJECTS:
    try:
        print(f"\n── {subject} ──────────────────────────────")
        df_raw  = loader.load_subject(subject)
        df      = loader.filter_target_states(df_raw)
        feat_df = process_subject_data(df)

        # Add Subject_ID column — required for LOSO grouping
        feat_df['Subject_ID'] = subject

        n_baseline = (feat_df['Label'] == 0).sum()
        n_stress   = (feat_df['Label'] == 1).sum()
        print(f"  Windows — Baseline: {n_baseline}, Stress: {n_stress}, Total: {len(feat_df)}")

        all_features.append(feat_df)

    except Exception as e:
        print(f"  ⚠️  Failed to process {subject}: {e}")
        failed.append(subject)

print(f"\n{'='*45}")
print(f"Processed: {len(all_features)} subjects")
if failed:
    print(f"Failed:    {failed}")


── S2 ──────────────────────────────
Loading S2 data... This might take a moment.
Extracting features using 60-second windows...
  Windows — Baseline: 19, Stress: 10, Total: 29

── S3 ──────────────────────────────
Loading S3 data... This might take a moment.
Extracting features using 60-second windows...
  Windows — Baseline: 19, Stress: 10, Total: 29

── S4 ──────────────────────────────
Loading S4 data... This might take a moment.
Extracting features using 60-second windows...
  Windows — Baseline: 19, Stress: 10, Total: 29

── S5 ──────────────────────────────
Loading S5 data... This might take a moment.
Extracting features using 60-second windows...
  Windows — Baseline: 20, Stress: 10, Total: 30

── S6 ──────────────────────────────
Loading S6 data... This might take a moment.
Extracting features using 60-second windows...
  Windows — Baseline: 20, Stress: 10, Total: 30

── S7 ──────────────────────────────
Loading S7 data... This might take a moment.
Extracting features using 6

In [9]:
full_df = pd.concat(all_features, ignore_index=True)

print(f"Total windows:   {len(full_df)}")
print(f"Total features:  {full_df.shape[1] - 2}")  # minus Label and Subject_ID
print(f"\nLabel distribution (all subjects):")
print(full_df['Label'].value_counts().sort_index().rename({0: 'Baseline', 1: 'Stress'}))
print(f"\nWindows per subject:")
print(full_df.groupby('Subject_ID').size().to_string())

Total windows:   451
Total features:  7

Label distribution (all subjects):
Label
Baseline    296
Stress      155
Name: count, dtype: int64

Windows per subject:
Subject_ID
S10    31
S11    31
S13    30
S14    30
S15    31
S16    30
S17    31
S2     29
S3     29
S4     29
S5     30
S6     30
S7     30
S8     30
S9     30


In [10]:
os.makedirs('../data', exist_ok=True)
full_df.to_csv('../data/features_all_subjects.csv', index=False)
print(f"Saved to ../data/features_all_subjects.csv")
print(f"Shape: {full_df.shape}")

Saved to ../data/features_all_subjects.csv
Shape: (451, 9)


In [11]:
results = train_and_evaluate_loso(full_df, save_dir='../models')

Preparing data for LOSO Cross-Validation...
Starting LOSO across 15 folds (one per subject)...

  Fold 01 | Subject S10 | Acc: 0.677 | F1: 0.286 | Windows: 31
  Fold 02 | Subject S11 | Acc: 0.806 | F1: 0.786 | Windows: 31
  Fold 03 | Subject S13 | Acc: 0.967 | F1: 0.952 | Windows: 30
  Fold 04 | Subject S14 | Acc: 0.833 | F1: 0.800 | Windows: 30
  Fold 05 | Subject S15 | Acc: 0.968 | F1: 0.952 | Windows: 31
  Fold 06 | Subject S16 | Acc: 1.000 | F1: 1.000 | Windows: 30
  Fold 07 | Subject S17 | Acc: 0.871 | F1: 0.833 | Windows: 31
  Fold 08 | Subject S2 | Acc: 0.621 | F1: 0.000 | Windows: 29
  Fold 09 | Subject S3 | Acc: 0.966 | F1: 0.947 | Windows: 29
  Fold 10 | Subject S4 | Acc: 0.690 | F1: 0.182 | Windows: 29
  Fold 11 | Subject S5 | Acc: 1.000 | F1: 1.000 | Windows: 30
  Fold 12 | Subject S6 | Acc: 0.933 | F1: 0.889 | Windows: 30
  Fold 13 | Subject S7 | Acc: 0.967 | F1: 0.947 | Windows: 30
  Fold 14 | Subject S8 | Acc: 0.967 | F1: 0.957 | Windows: 30
  Fold 15 | Subject S9 | Acc:

In [12]:
per_subject_df = pd.DataFrame(results['per_subject'])
per_subject_df = per_subject_df.sort_values('accuracy', ascending=False).reset_index(drop=True)

print("Per-Subject Results:")
print(per_subject_df.to_string(index=False))
print(f"\nBest subject:  {per_subject_df.iloc[0]['subject']}  "
      f"(Acc: {per_subject_df.iloc[0]['accuracy']:.3f})")
print(f"Worst subject: {per_subject_df.iloc[-1]['subject']}  "
      f"(Acc: {per_subject_df.iloc[-1]['accuracy']:.3f})")

Per-Subject Results:
subject  accuracy    f1  n_windows
    S16     1.000 1.000         30
     S5     1.000 1.000         30
    S15     0.968 0.952         31
     S7     0.967 0.947         30
     S8     0.967 0.957         30
    S13     0.967 0.952         30
     S3     0.966 0.947         29
     S6     0.933 0.889         30
     S9     0.900 0.842         30
    S17     0.871 0.833         31
    S14     0.833 0.800         30
    S11     0.806 0.786         31
     S4     0.690 0.182         29
    S10     0.677 0.286         31
     S2     0.621 0.000         29

Best subject:  S16  (Acc: 1.000)
Worst subject: S2  (Acc: 0.621)
